## Step 0: Load libraries

In [58]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

## Step 1: Load and Clean Data

Three sequential cleaning operations:

1. **Drop nulls** — rows with any missing value are removed (primarily affects amenity distance columns where geocoding returned no result: 157,821 → 114,147, dropped 43,674)
2. **Drop duplicates** — remove rows with duplicated data

In [59]:
df_raw = pd.read_csv("../../merged_data/[FINAL]hdb_with_amenities_macro_pre2026.csv")
print(f"Initial shape: {df_raw.shape}")

df = df_raw.dropna()
print(f"After dropping nulls: {df.shape} (dropped {len(df_raw) - len(df)})")

# Remove duplicate rows to avoid repeated transactions affecting model training
n_before = len(df)
df = df.drop_duplicates()
print(f"After dropping duplicates: {df.shape} (dropped {n_before - len(df)})")

# 2020 is excluded due to the COVID-19 structural break which caused abnormal market conditions
# unrepresentative of typical pricing relationships.
df["year_temp"] = pd.to_datetime(df["month"]).dt.year
n_before = len(df)
df = df[df["year_temp"] != 2020].drop(columns="year_temp")
print(f"After dropping 2020: {df.shape} (dropped {n_before - len(df)})")

df['log_resale_price_real'] = np.log(df['resale_price_real'])  # Log-transform target — preserves resale_price_real for metric computation

Initial shape: (134479, 37)
After dropping nulls: (134301, 37) (dropped 178)
After dropping duplicates: (134211, 37) (dropped 90)
After dropping 2020: (134211, 37) (dropped 0)


## Step 2: Stratified Train/Validation Split (80/20)
***Match baseline OLS model for comparability***

**Target variable:** `log_resale_price_real` — using RPI-adjusted prices means the model captures structural flat-level pricing drivers rather than market-wide appreciation, since `year` is excluded as a feature.

**80/20 ratio:** With ~97,000 transactions, yields ~77,000 training and ~19,000 validation rows. The training set is large enough for stable estimation; the validation set is large enough for statistically reliable RMSE and linlin loss estimates. A 70/30 split is only preferred for datasets of a few thousand rows.

**Stratification key: `town + flat_type + year`** — three reasons:
1. **Town** — 26 HDB towns with substantially different price levels; random sampling could over-represent expensive towns in training.
2. **Flat type** — Highly imbalanced (4 ROOM ~42% vs 2 ROOM ~2%); stratifying prevents minority types being underrepresented.
3. **Year** — Despite RPI adjustment, residual structural differences exist across years (post-COVID demand surge, cooling measures); ensures proportional representation of every year in both sets.

In [60]:
import re

# Target variable: log_resale_price_real
# RPI-adjusted prices are used so the model learns purely from flat characteristics
# rather than market-wide price appreciation over time, since year is not a model feature.
target = "log_resale_price_real"


from sklearn.model_selection import train_test_split

df["strat_key"] = (df["town"].astype(str) + "_" +
                   df["flat_type"].astype(str) + "_" +
                   df["year"].astype(str))

strat_counts = df["strat_key"].value_counts()
valid_keys = strat_counts[strat_counts >= 2].index
n_before = len(df)
df = df[df["strat_key"].isin(valid_keys)]
print(f"Dropped {n_before - len(df)} rows with singleton strat_key combinations. Remaining: {len(df):,}")

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["strat_key"])
print(f"Train size: {len(train_df):,} | Test size: {len(val_df):,}")

print("\nYear distribution (%):")
train_year = train_df["year"].value_counts(normalize=True).sort_index().rename("Train %")
test_year = val_df["year"].value_counts(normalize=True).sort_index().rename("Validation %")
year_dist = pd.concat([train_year, test_year], axis=1)
print(year_dist.applymap(lambda x: f"{x:.2%}"))

print("\nFlat type distribution (%):")
train_flat = train_df["flat_type"].value_counts(normalize=True).rename("Train %")
test_flat = val_df["flat_type"].value_counts(normalize=True).rename("Validation %")
flat_dist = pd.concat([train_flat, test_flat], axis=1)
print(flat_dist.applymap(lambda x: f"{x:.2%}"))

Dropped 6 rows with singleton strat_key combinations. Remaining: 134,205
Train size: 107,364 | Test size: 26,841

Year distribution (%):
     Train % Validation %
2021  21.61%       21.60%
2022  19.87%       19.86%
2023  19.15%       19.16%
2024  20.71%       20.70%
2025  18.67%       18.68%

Flat type distribution (%):
                 Train % Validation %
4 ROOM            43.02%       43.01%
5 ROOM            24.25%       24.23%
3 ROOM            23.71%       23.71%
EXECUTIVE          6.62%        6.62%
2 ROOM             2.35%        2.36%
1 ROOM             0.03%        0.03%
MULTI-GENERATION   0.03%        0.03%


## Step 3: Random Forest Model
***Keep feature set aligned with baseline OLS model***

**Feature matrix:**
| Type | Features | Treatment |
|------|----------|-----------|
| Continuous (10) | `remaining_lease_years`, `nearest_train_dist_m`, `num_primary_1km`, `num_parks_1km`, 6 amenity distances | Fit on train only |
| Categorical — flat type (4) | 3 ROOM, 4 ROOM, 5 ROOM, EXECUTIVE | One-hot; reference = 2 ROOM |
| Categorical — town (25) | All towns except ANG MO KIO | One-hot; reference = ANG MO KIO |
| Categorical — floor (2) | Mid, High | One-hot; reference = Low (floors 1–5) |

**Excluded features:**
- `floor_area_sqm` — not a user-facing input at inference time; users select flat type but do not enter exact sqm. Flat type dummies implicitly capture size.

In [61]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
import numpy as np

# =========================================
# Features 
# =========================================
continuous_features = [
    "remaining_lease_years", "nearest_train_dist_m",
    "dist_nearest_hawker_m", "dist_nearest_primary_m", "num_primary_1km",
    "dist_nearest_park_m", "num_parks_1km", "dist_nearest_sportsg_m",
    "dist_nearest_mall_m", "dist_nearest_healthcare_m"  
]

categorical_features = ["flat_type", "town", "floor_category"]

# =========================================
# One-hot encoding 
# =========================================
train_encoded = pd.get_dummies(train_df, columns=categorical_features, drop_first=False)
val_encoded = pd.get_dummies(val_df, columns=categorical_features, drop_first=False)

# Identify dummy columns
dummy_cols = [c for c in train_encoded.columns
              if any(c.startswith(f"{f}_") for f in categorical_features)]

# =========================================
# Combine features 
# =========================================
X_train = pd.concat([train_encoded[continuous_features],
                     train_encoded[dummy_cols].astype(int)], axis=1)

X_val = pd.concat([val_encoded[continuous_features],
                    val_encoded[dummy_cols].astype(int)], axis=1)

# Align validation columns to train
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)

# Targets
y_train = train_df[target]          
y_val = val_df[target]

y_train_actual = train_df['resale_price_real']
y_val_actual = val_df['resale_price_real']

# =========================================
# Train Random Forest
# =========================================
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# =========================================
# Predictions
# =========================================
y_val_pred_log = rf_model.predict(X_val)
y_val_pred = np.exp(y_val_pred_log)   # convert back to price

# =========================================
# Evaluation 
# =========================================
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2))

def linlin_loss(y_true, y_pred, underpredict_weight=2.0):
    errors = np.array(y_true) - np.array(y_pred)
    loss = np.where(errors > 0, underpredict_weight * errors, -errors)
    return np.mean(loss)

validation_rmse = rmse(y_val_actual, y_val_pred)
validation_linlin = linlin_loss(y_val_actual, y_val_pred, underpredict_weight=2.0)
r2 = r2_score(y_val_actual, y_val_pred)

print("=== RANDOM FOREST MODEL EVALUATION ===")
print(f"R^2: {r2:.3f}")
print(f"RMSE: ${validation_rmse:,.0f}")
print(f"Linlin Loss: ${validation_linlin:,.0f}")


=== RANDOM FOREST MODEL EVALUATION ===
R^2: 0.964
RMSE: $39,397
Linlin Loss: $41,145


## Step 4: Load and Train Test Data (2026 data)
***Test using 2026 data, compare differences between actual and predicted prices***

In [62]:
df_test_raw = pd.read_csv("../../merged_data/[FINAL]hdb_with_amenities_macro_2026.csv")
print(f"Initial shape: {df_test_raw.shape}")

df_test = df_test_raw.dropna()
print(f"After dropping nulls: {df_test.shape} (dropped {len(df_test_raw) - len(df_test)})")

# Remove duplicate rows to avoid repeated transactions affecting model training
n_before = len(df_test)
df_test = df_test.drop_duplicates()
print(f"After dropping duplicates: {df_test.shape} (dropped {n_before - len(df_test)})")

df_test['log_resale_price_real'] = np.log(df_test['resale_price_real'])  # Log-transform target — preserves resale_price_real for metric computation

Initial shape: (6058, 37)
After dropping nulls: (6055, 37) (dropped 3)
After dropping duplicates: (6051, 37) (dropped 4)


## Step 5: Check results for test data (out-of-sample data)

In [63]:

# =========================================
# One-hot encoding
# =========================================
df_test_encoded = pd.get_dummies(df_test, columns=categorical_features, drop_first=False)

# =========================================
# Build feature matrix
# =========================================
X_test = pd.concat([df_test_encoded[continuous_features],
                     df_test_encoded[dummy_cols].astype(int)], axis=1)


# =========================================
# Align columns with training data
# =========================================
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# =========================================
# Predictions
# =========================================
y_test_pred_log = rf_model.predict(X_test)
y_test_pred = np.exp(y_test_pred_log)

# =========================================
# Actual values
# =========================================
y_test_actual = df_test['resale_price_real']

# =========================================
# Evaluation
# =========================================
rmse_test = rmse(y_test_actual, y_test_pred)
linlin_test = linlin_loss(y_test_actual, y_test_pred, underpredict_weight=2.0)
r2_test = r2_score(y_test_actual, y_test_pred)

print("=== 2026 TEST SET (OUT-OF-SAMPLE) ===")
print(f"R^2: {r2_test:.3f}")
print(f"RMSE: ${rmse_test:,.0f}")
print(f"Linlin Loss: ${linlin_test:,.0f}")



=== 2026 TEST SET (OUT-OF-SAMPLE) ===
R^2: 0.960
RMSE: $41,784
Linlin Loss: $42,257


## Step 6: Random Forest Model #2 (with extra variables)
***Additional features added - "dist_cbd_m", "floor_area_sqm"***

**Feature matrix:**
| Type | Features | Treatment |
|------|----------|-----------|
| Continuous (12) | `remaining_lease_years`, `nearest_train_dist_m`, `num_primary_1km`, `num_parks_1km`, `dist_cbd_m`, `floor_area_sqm`, 6 amenity distances | Fit on train only |
| Categorical — flat type (4) | 3 ROOM, 4 ROOM, 5 ROOM, EXECUTIVE | One-hot; reference = 2 ROOM |
| Categorical — town (25) | All towns except ANG MO KIO | One-hot; reference = ANG MO KIO |
| Categorical — floor (2) | Mid, High | One-hot; reference = Low (floors 1–5) |

**Excluded features:**
- `floor_area_sqm` — not a user-facing input at inference time; users select flat type but do not enter exact sqm. Flat type dummies implicitly capture size.

In [64]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
import numpy as np

# =========================================
# Features 
# =========================================
continuous_features = [
    "remaining_lease_years", "nearest_train_dist_m",
    "dist_nearest_hawker_m", "dist_nearest_primary_m", "num_primary_1km",
    "dist_nearest_park_m", "num_parks_1km", "dist_nearest_sportsg_m",
    "dist_nearest_mall_m", "dist_nearest_healthcare_m", 
    "dist_cbd_m", "floor_area_sqm" 
]

categorical_features = ["flat_type", "town", "floor_category"]

# =========================================
# One-hot encoding 
# =========================================
train_encoded = pd.get_dummies(train_df, columns=categorical_features, drop_first=False)
val_encoded = pd.get_dummies(val_df, columns=categorical_features, drop_first=False)

# Identify dummy columns
dummy_cols = [c for c in train_encoded.columns
              if any(c.startswith(f"{f}_") for f in categorical_features)]

# =========================================
# Combine features 
# =========================================
X_train = pd.concat([train_encoded[continuous_features],
                     train_encoded[dummy_cols].astype(int)], axis=1)

X_val = pd.concat([val_encoded[continuous_features],
                    val_encoded[dummy_cols].astype(int)], axis=1)

# Align validation columns to train
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)

# Targets
y_train = train_df[target]          
y_val = val_df[target]

y_train_actual = train_df['resale_price_real']
y_val_actual = val_df['resale_price_real']

# =========================================
# Train Random Forest
# =========================================
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# =========================================
# Predictions
# =========================================
y_val_pred_log = rf_model.predict(X_val)
y_val_pred = np.exp(y_val_pred_log)   # convert back to price

# =========================================
# Evaluation 
# =========================================
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2))

def linlin_loss(y_true, y_pred, underpredict_weight=2.0):
    errors = np.array(y_true) - np.array(y_pred)
    loss = np.where(errors > 0, underpredict_weight * errors, -errors)
    return np.mean(loss)

validation_rmse = rmse(y_val_actual, y_val_pred)
validation_linlin = linlin_loss(y_val_actual, y_val_pred, underpredict_weight=2.0)
r2 = r2_score(y_val_actual, y_val_pred)

print("=== RANDOM FOREST MODEL EVALUATION ===")
print(f"R^2: {r2:.3f}")
print(f"RMSE: ${validation_rmse:,.0f}")
print(f"Linlin Loss: ${validation_linlin:,.0f}")


=== RANDOM FOREST MODEL EVALUATION ===
R^2: 0.971
RMSE: $35,425
Linlin Loss: $37,624


## Step 7: Check results for test data (out-of-sample data)

In [65]:
# =========================================
# One-hot encoding
# =========================================
df_test_encoded = pd.get_dummies(df_test, columns=categorical_features, drop_first=False)

# =========================================
# Build feature matrix
# =========================================
X_test = pd.concat([df_test_encoded[continuous_features],
                     df_test_encoded[dummy_cols].astype(int)], axis=1)


# =========================================
# Align columns with training data
# =========================================
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# =========================================
# Predictions
# =========================================
y_test_pred_log = rf_model.predict(X_test)
y_test_pred = np.exp(y_test_pred_log)

# =========================================
# Actual values
# =========================================
y_test_actual = df_test['resale_price_real']

# =========================================
# Evaluation
# =========================================
rmse_test = rmse(y_test_actual, y_test_pred)
linlin_test = linlin_loss(y_test_actual, y_test_pred, underpredict_weight=2.0)
r2_test = r2_score(y_test_actual, y_test_pred)

print("=== 2026 TEST SET (OUT-OF-SAMPLE) ===")
print(f"R^2: {r2_test:.3f}")
print(f"RMSE: ${rmse_test:,.0f}")
print(f"Linlin Loss: ${linlin_test:,.0f}")



=== 2026 TEST SET (OUT-OF-SAMPLE) ===
R^2: 0.967
RMSE: $38,039
Linlin Loss: $38,544


## Step 8: Conformal Prediction Intervals

**Conformal prediction gives us a rigorous way to wrap uncertainty around our point estimates. The key guarantee: if the 2026 data comes from the same distribution as our validation set, then (1 - alpha)% of our intervals will contain the true price.**

**We use the validation split as our "calibration set" — data the model has never seen during training, so its errors are honest.**

In [68]:
# ===================================================================
# Step 1: Calibration residuals (in actual price space, after exp) 
# ===================================================================
# SIGNED residuals (true - pred), not absolute. Sign matters because we treat underprediction and overprediction differently.
# We compute these AFTER exp(), in actual dollar space.
# If we worked in log space, our intervals would be multiplicative and much harder to interpret or apply to new predictions.

cal_residuals = y_val_actual.values - y_val_pred   # true - pred

# ===================================================================
# Step 2: Asymmetric quantiles to match linlin loss 
# ===================================================================
# Because underprediction costs 2x more (underpredict_weight=2.0), we split the alpha "miss budget" unequally between the two tails.
# The intuition: if we have 5% of predictions allowed to miss, and underprediction is twice as bad, we protect the upper side twice as hard — giving it only 1/(1+2) = 33% of the miss budget.
# upper tail gets: alpha * 1/(1+w)    = 0.05 * 0.333 = 1.67% tolerance
# lower tail gets: alpha * w/(1+w)    = 0.05 * 0.667 = 3.33% tolerance
# This means our upper bound is held very tight (only 1.67% of true prices are allowed to exceed it), while the lower bound is looser.
alpha = 0.05   # 95% overall coverage
w = 2.0        # underpredict_weight

alpha_upper = alpha * (1 / (1 + w))    # 0.0167 — upper tail tolerance
alpha_lower = alpha * (w / (1 + w))    # 0.0333 — lower tail tolerance

n = len(cal_residuals)

# Finite-sample correction: use (n/(n+1)) instead of plain quantile.
# Without this, coverage is only guaranteed asymptotically (large n).
# With it, we get exact finite-sample coverage — always include this.
q_low  = np.quantile(cal_residuals, alpha_lower * (n / (n + 1)))   # negative: how much we can overpredict
q_high = np.quantile(cal_residuals, 1 - alpha_upper * (n / (n + 1)))  # positive: how much we can underpredict

print(f"\n=== CONFORMAL PREDICTION SETUP ===")
print(f"Calibration samples (n): {n}")
print(f"Lower quantile (q_low):  ${q_low:,.0f}  (model can overpredict by this much)")
print(f"Upper quantile (q_high): ${q_high:,.0f}  (model can underpredict by this much)")
print(f"Typical interval width:  ${q_high - q_low:,.0f}")

# --- Step 3: Diagnosis on validation set ---
# Before applying to 2026, sanity-check that the intervals behave correctly on the calibration set itself.
# Expected: empirical coverage ≈ 95%, upper misses ≈ 1.67%, lower misses ≈ 3.33%.
lower_val = y_val_pred + q_low    # q_low is negative, so this subtracts
upper_val = y_val_pred + q_high

covered_val = (
    (y_val_actual.values >= lower_val) &
    (y_val_actual.values <= upper_val)
)

# Count directional misses separately — they mean different things.
# Upper misses = we underpredicted (costly, per our linlin loss).
# Lower misses = we overpredicted (less costly, so we allow more of these).
underpredict_misses = (y_val_actual.values > upper_val).sum()
overpredict_misses  = (y_val_actual.values < lower_val).sum()

print(f"\n=== CALIBRATION DIAGNOSTICS ===")
print(f"Empirical coverage:       {covered_val.mean():.2%}  (target: {1-alpha:.0%})")
print(f"Underpredict misses:      {underpredict_misses} ({underpredict_misses/n:.2%})  (target: {alpha_upper:.2%})")
print(f"Overpredict misses:       {overpredict_misses} ({overpredict_misses/n:.2%})  (target: {alpha_lower:.2%})")


# =========================================
# Apply conformal intervals to 2026 OOS Data
# =========================================
# q_low and q_high are fixed constants from the calibration step above.
# Every 2026 prediction gets the same additive shift applied.
# The interval is intentionally asymmetric: wider above the point estimate (by q_high) than below (by |q_low|), because we penalise underprediction more.
lower_oos = y_test_pred + q_low
upper_oos = y_test_pred + q_high

covered_oos = (
    (y_test_actual.values >= lower_oos) &
    (y_test_actual.values <= upper_oos)
)

under_oos = (y_test_actual.values > upper_oos).sum()
over_oos  = (y_test_actual.values < lower_oos).sum()
n_oos     = len(y_test_actual)


print(f"\n=== OOS (2026) CONFORMAL INTERVAL PERFORMANCE ===")
# If OOS coverage is well below 95%: distribution shift between <=2025 and 2026 (e.g. cooling measures, rate changes). 
# If OOS coverage is well above 95%: intervals are too conservative (validation residuals were larger than typical OOS errors). Could tighten alpha to 0.08-0.10 and still have valid coverage.
print(f"Empirical coverage:   {covered_oos.mean():.2%}  (target: 95%)")
print(f"Underpredict misses:  {under_oos} ({under_oos/n_oos:.2%})  (target: {alpha_upper:.2%})")
print(f"Overpredict misses:   {over_oos} ({over_oos/n_oos:.2%})  (target: {alpha_lower:.2%})")


# --- Final output dataframe ---
results_oos = pd.DataFrame({
    'actual_price':  y_test_actual.values,
    'pred_price':    y_test_pred,
    'lower_95':      lower_oos,
    'upper_95':      upper_oos,
    'covered':       covered_oos,
    'residual':      y_test_actual.values - y_test_pred
})

print(f"\n=== SAMPLE PREDICTIONS WITH INTERVALS ===")
print(results_oos.head(10).to_string(index=False))


=== CONFORMAL PREDICTION SETUP ===
Calibration samples (n): 26841
Lower quantile (q_low):  $-60,570  (model can overpredict by this much)
Upper quantile (q_high): $86,858  (model can underpredict by this much)
Typical interval width:  $147,428

=== CALIBRATION DIAGNOSTICS ===
Empirical coverage:       95.00%  (target: 95%)
Underpredict misses:      448 (1.67%)  (target: 1.67%)
Overpredict misses:       895 (3.33%)  (target: 3.33%)

=== OOS (2026) CONFORMAL INTERVAL PERFORMANCE ===
Empirical coverage:   93.93%  (target: 95%)
Underpredict misses:  116 (1.92%)  (target: 1.67%)
Overpredict misses:   251 (4.15%)  (target: 3.33%)

=== SAMPLE PREDICTIONS WITH INTERVALS ===
 actual_price    pred_price      lower_95      upper_95  covered      residual
     345000.0 321372.892931 260802.807023 408230.878358     True  23627.107069
     325000.0 321086.992523 260516.906616 407944.977950     True   3913.007477
     330000.0 333969.112115 273399.026207 420827.097542     True  -3969.112115
     658